In [0]:
# ─── Stage 3 — Production refactor 
# Logic extracted to src/market_pulse/bronze.py
#
# BEFORE (v1 — Stage 2):
#   json_schema and process_batch defined inline in the notebook
#
# AFTER (v2 — Stage 3):
#   from market_pulse.bronze import json_schema, process_batch
#
# The notebook is now a thin orchestration layer.
# All transformation logic lives in src/market_pulse/bronze.py
# 


import sys
sys.path.insert(0, "/Workspace/Repos/martalimas@gmail.com/market-pulse-pipeline/src")

from market_pulse.config import (
    BRONZE_LANDING_PATH,
    BRONZE_INGESTION_PATH,
    BRONZE_CHECKPOINT_PATH
)
from market_pulse.bronze import json_schema, process_batch

print("✅ Módulos importados de src/market_pulse/")
print(f"✅ Landing:    {BRONZE_LANDING_PATH}")
print(f"✅ Target:     {BRONZE_INGESTION_PATH}")
print(f"✅ Checkpoint: {BRONZE_CHECKPOINT_PATH}")

In [0]:
#  Auto Loader Stream Toda a complexidade (schema, explode, flatten) está agora em bronze.py

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", BRONZE_CHECKPOINT_PATH)
    .option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")
    .option("cloudFiles.inferColumnTypes", "false")
    .schema(json_schema)
    .load(BRONZE_LANDING_PATH)
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", BRONZE_CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)